# Lab 03 — Subqueries e CTEs (DuckDB)

**Onde roda:** 🟢 Browser (JupyterLite). Execute célula a célula.

Objetivo: compor consultas em etapas com subqueries e `WITH` (CTEs).

In [ ]:
try:
    import duckdb
except ModuleNotFoundError:
    import piplite; await piplite.install('duckdb'); import duckdb
import pandas as pd
pedidos = pd.DataFrame([
    (1,'SP','eletronicos',1200.0,1),(2,'SP','livros',50.0,2),(3,'RJ','livros',30.0,1),
    (4,'MG','casa',80.0,3),(5,'SP','eletronicos',800.0,2),(6,'RJ','casa',150.0,4),
    (7,'SP','livros',45.0,1),(8,'MG','eletronicos',600.0,3),(9,'RJ','eletronicos',900.0,2),
    (10,'SP','casa',200.0,5),(11,'MG','livros',25.0,4),(12,'SP','eletronicos',1500.0,1),
    (13,'RJ','livros',60.0,5),(14,'MG','casa',120.0,3),(15,'SP','livros',40.0,2),
], columns=['id','estado','categoria','valor','cliente_id'])
print('média de valor:', round(pedidos['valor'].mean(), 2))

## 1. Subquery escalar: pedidos acima da média

In [ ]:
duckdb.query('''
    SELECT id, valor
    FROM pedidos
    WHERE valor > (SELECT AVG(valor) FROM pedidos)
    ORDER BY valor DESC
''').to_df()

## 2. CTE: total por cliente, em duas etapas

In [ ]:
duckdb.query('''
    WITH total_por_cliente AS (
        SELECT cliente_id, SUM(valor) AS total
        FROM pedidos GROUP BY cliente_id
    )
    SELECT * FROM total_por_cliente ORDER BY total DESC
''').to_df()

## 3. Sua vez (mini-desafio)
Usando uma CTE, traga os clientes cujo **total** é maior que a **média dos totais** (colunas `cliente_id`, `total`), ordem desc. Verifique.

In [ ]:
resposta = duckdb.query('''
    WITH t AS (SELECT cliente_id, SUM(valor) AS total FROM pedidos GROUP BY cliente_id)
    SELECT cliente_id, total FROM t
    WHERE total > (SELECT AVG(total) FROM t)
    ORDER BY total DESC
''').to_df()
resposta

In [ ]:
def verificar(df):
    try:
        assert list(df['cliente_id']) == [1, 2], 'Deveriam ser os clientes 1 e 2 (acima da média de gasto).'
        print('\u2705 Correto! CTE + subquery na comparação.')
    except AssertionError as e:
        print('\u274c', e)

verificar(resposta)